In [ ]:
"""
Success Criteria

completed project should:

- Handle 1,000+ documents reliably, and report measured query latency
  (the focus is correctness + evaluation, not chasing speed at small scale)
- Demonstrate that hybrid search improves results (or document why it doesn't)
- Include rich metadata that enables meaningful filtering
- Show a clear evaluation methodology with test queries
- Provide comprehensive documentation that someone else could follow
- Present technical decisions with justified reasoning
"""

In [ ]:
"""
The Guardian - Data Collection Script

What it does:
- Pulls ~2,000 recent Guardian articles across several sections.
- Saves:
  - guardian_documents.csv
  - guardian_documents_full.json

Setup:
1) Get an API key: https://open-platform.theguardian.com/access/
2) pip install requests pandas tqdm python-dotenv
3) Create a .env file in this folder:
   GUARDIAN_API_KEY=your_key_here
4) Run:
   python collect_guardian_data.py

Notes:
- Uses a stable string id (Guardian's content id).
- Writes a small progress file so you can resume if the run gets interrupted.
"""

In [ ]:
import os
import json
import time
import string
import hashlib
from pathlib import Path
from datetime import datetime, timedelta, timezone


import nltk
import psycopg2
import requests
import numpy as np
import pandas as pd
from tqdm import tqdm
from cohere import ClientV2
from dotenv import load_dotenv
from rank_bm25 import BM25Okapi
from nltk.tokenize import sent_tokenize
from pgvector.psycopg2 import register_vector

/Users/sbamwite/Desktop/rags/env/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
# -----------------------------
# CONFIG
# -----------------------------
load_dotenv()
GUARDIAN_API_KEY = os.getenv("GUARDIAN_API_KEY")

if not GUARDIAN_API_KEY:
    raise ValueError(
        "Missing GUARDIAN_API_KEY in .env.\n"
        "Get a key at https://open-platform.theguardian.com/access/\n"
        "Then create .env with: GUARDIAN_API_KEY=your_key_here"
    )

BASE_URL = "https://content.guardianapis.com/search"

SECTIONS = ["politics", "technology", "science", "sport", "culture"]
ARTICLES_PER_SECTION = 400  # 5 * 400 = 2,000

# Guardian allows up to 200 per page in many cases, but 50 is conservative and reliable.
PAGE_SIZE = 50

# Date window (recent content tends to be friendlier for evaluation queries)
DAYS_BACK = 180

# Polite pacing
SLEEP_BETWEEN_REQUESTS_SEC = 0.25

# Resume support
PROGRESS_PATH = Path("guardian_progress.json")

In [4]:
# -----------------------------
# HELPERS
# -----------------------------
def utc_now():
    return datetime.now(timezone.utc)

def iso_date(dt: datetime) -> str:
    return dt.strftime("%Y-%m-%d")

def safe_int(x, default=0):
    try:
        return int(x)
    except Exception:
        return default

def stable_row_id(guardian_id: str) -> str:
    # Guardian IDs are already stable; this is just a fallback if needed.
    if guardian_id:
        return guardian_id
    return "guardian_" + hashlib.sha1(os.urandom(16)).hexdigest()[:16]

def load_progress():
    if not PROGRESS_PATH.exists():
        return {"completed_sections": [], "rows": []}
    with open(PROGRESS_PATH, "r", encoding="utf-8") as f:
        return json.load(f)

def save_progress(state):
    with open(PROGRESS_PATH, "w", encoding="utf-8") as f:
        json.dump(state, f, indent=2, ensure_ascii=False)

def guardian_request(params, session: requests.Session, max_retries=5):
    """
    Basic retry with backoff.
    - 429: wait and retry
    - network blips: retry
    """
    backoff = 2.0
    for attempt in range(1, max_retries + 1):
        try:
            r = session.get(BASE_URL, params=params, timeout=20)
            if r.status_code == 200:
                return r.json()

            # Rate limit / burst protection
            if r.status_code == 429:
                wait = min(60, int(backoff * attempt))
                print(f"⚠️  Rate limited (429). Sleeping {wait}s then retrying...")
                time.sleep(wait)
                continue

            # Other API errors: show message if present, then stop retrying.
            try:
                msg = r.json().get("response", {}).get("message") or r.json().get("message")
            except Exception:
                msg = None
            print(f"⚠️  API error {r.status_code}. {msg or ''}".strip())
            return None

        except requests.RequestException as e:
            if attempt == max_retries:
                print(f"❌ Network error after {max_retries} tries: {e}")
                return None
            wait = min(30, int(backoff * attempt))
            print(f"⚠️  Network error: {e}. Sleeping {wait}s then retrying...")
            time.sleep(wait)

    return None

def normalize_guardian_result(item: dict) -> dict:
    """
    Converts a Guardian API result into the unified row schema.
    Skips items without usable body text.
    """
    fields = item.get("fields") or {}
    body = fields.get("bodyText") or ""
    body = body.strip()

    if not body:
        return None

    tags = item.get("tags") or []
    tag_names = [t.get("webTitle") for t in tags if t.get("webTitle")]
    tags_str = ", ".join(tag_names) if tag_names else ""

    pub = item.get("webPublicationDate")  # ISO timestamp string
    title = item.get("webTitle") or ""

    row = {
        "id": stable_row_id(item.get("id")),
        "title": title,
        "body_text": body,
        "source": "the_guardian",
        "category": (item.get("sectionName") or item.get("sectionId") or "").strip() or "unknown",
        "publication_date": pub or "",
        "author": (fields.get("byline") or "Unknown").strip(),
        "url": item.get("webUrl") or "",
        "word_count": safe_int(fields.get("wordcount"), default=len(body.split())),
        "summary": (fields.get("trailText") or "").strip(),
        "tags": tags_str,
    }
    return row

def fetch_section(section: str, start_date: datetime, end_date: datetime, target_n: int, session: requests.Session):
    rows = []
    pages_needed = (target_n + PAGE_SIZE - 1) // PAGE_SIZE

    for page in tqdm(range(1, pages_needed + 1), desc=f"{section}"):
        params = {
            "api-key": GUARDIAN_API_KEY,
            "section": section,
            "from-date": iso_date(start_date),
            "to-date": iso_date(end_date),
            "page-size": PAGE_SIZE,
            "page": page,
            "order-by": "newest",
            "show-fields": "bodyText,wordcount,byline,trailText",
            "show-tags": "keyword",
        }

        data = guardian_request(params, session=session)
        print(f"--------Sleeping for 2 seconds after call--------")
        time.sleep(2)
        if not data:
            break

        results = (data.get("response") or {}).get("results") or []
        if not results:
            break

        for item in results:
            row = normalize_guardian_result(item)
            if row:
                rows.append(row)

        # Polite pacing
        time.sleep(SLEEP_BETWEEN_REQUESTS_SEC)

        if len(rows) >= target_n:
            break

    # Guardian can occasionally return duplicates across pages if content changes quickly;
    # de-dupe on id to keep things tidy.
    dedup = {}
    for r in rows:
        dedup[r["id"]] = r

    out = list(dedup.values())
    out.sort(key=lambda x: x.get("publication_date", ""), reverse=True)
    return out[:target_n]


In [ ]:
# 1 - COLLECT DATA

def main():
    end_date = utc_now()
    start_date = end_date - timedelta(days=DAYS_BACK)

    print("=" * 70)
    print("The Guardian - Article Collection")
    print("=" * 70)
    print(f"Sections: {', '.join(SECTIONS)}")
    print(f"Target: {ARTICLES_PER_SECTION * len(SECTIONS):,} articles")
    print(f"Date range: {iso_date(start_date)} to {iso_date(end_date)} (UTC)")
    print("=" * 70)

    state = load_progress()
    completed_sections = set(state.get("completed_sections", []))
    all_rows = state.get("rows", [])

    # For fast lookups + de-dupe while resuming
    by_id = {r["id"]: r for r in all_rows if r.get("id")}

    with requests.Session() as session:
        for section in SECTIONS:
            if section in completed_sections:
                print(f"✓ Skipping {section} (already collected)")
                continue

            print(f"\nCollecting section: {section}")
            section_rows = fetch_section(
                section=section,
                start_date=start_date,
                end_date=end_date,
                target_n=ARTICLES_PER_SECTION,
                session=session,
            )

            for r in section_rows:
                by_id[r["id"]] = r

            completed_sections.add(section)

            # Save progress after each section so an interruption is painless.
            state = {
                "completed_sections": sorted(list(completed_sections)),
                "rows": list(by_id.values()),
            }
            save_progress(state)

            print(f"✓ Collected {len(section_rows)} rows for {section}")

    # Final dataset
    rows = list(by_id.values())

    # Sort newest-first for nicer browsing
    rows.sort(key=lambda x: x.get("publication_date", ""), reverse=True)

    df = pd.DataFrame(rows)

    csv_path = "guardian_documents.csv"
    json_path = "guardian_documents_full.json"

    df.to_csv(csv_path, index=False)
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(rows, f, indent=2, ensure_ascii=False)

    # Clean up progress file after success (keeps the folder tidy)
    if PROGRESS_PATH.exists():
        PROGRESS_PATH.unlink()

    print("\n" + "=" * 70)
    print("DONE")
    print("=" * 70)
    print(f"✓ Saved: {csv_path}")
    print(f"✓ Saved: {json_path}")

    if not df.empty:
        print("\nDataset summary:")
        print(df["category"].value_counts().to_string())
        print(f"\nAvg word_count: {df['word_count'].mean():.0f}")
        print(f"Word_count range: {df['word_count'].min():.0f}–{df['word_count'].max():.0f}")

In [6]:
main()

The Guardian - Article Collection
Sections: politics, technology, science, sport, culture
Target: 2,000 articles
Date range: 2025-10-10 to 2026-04-08 (UTC)



politics:   0%|          | 0/8 [00:00<?, ?it/s]

--------Sleeping for 2 seconds after call--------


politics:  12%|█▎        | 1/8 [00:03<00:27,  3.92s/it]

--------Sleeping for 2 seconds after call--------


...

--------Sleeping for 2 seconds after call--------


culture:  88%|████████▊ | 7/8 [00:18<00:02,  2.61s/it]

--------Sleeping for 2 seconds after call--------


culture: 100%|██████████| 8/8 [00:20<00:00,  2.59s/it]


✓ Collected 399 rows for culture

DONE
✓ Saved: guardian_documents.csv
✓ Saved: guardian_documents_full.json

Dataset summary:
category
Politics      400
Technology    400
Culture       399
Sport         396
Science       301

Avg word_count: 1013
Word_count range: 62–17451


In [7]:
df = pd.read_csv('guardian_documents.csv')
print(f"Collected {len(df)} documents")
print("\nDocuments by category:")
print(df['category'].value_counts())
print(f"\nAverage word count: {df['word_count'].mean():.0f} words")

Collected 1896 documents

Documents by category:
category
Politics      400
Technology    400
Culture       399
Sport         396
Science       301
Name: count, dtype: int64

Average word count: 1013 words


In [10]:
nltk.download('punkt')

# nltk.download('punkt', quiet=True)

[nltk_data] Downloading package punkt to /Users/sbamwite/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [11]:
def chunk_by_sentences(text, target_words=500, min_words=100):
    """
    Chunk text by grouping sentences to target word count.

    Args:
        text: Document text to chunk
        target_words: Target words per chunk (default 500)
        min_words: Minimum words for valid chunk (default 100)

    Returns:
        List of text chunks
    """
    sentences = sent_tokenize(text)
    chunks = []
    current_chunk = []
    current_count = 0

    for sentence in sentences:
        sentence_words = len(sentence.split())

        # Save current chunk if adding this sentence would exceed target
        if current_count > 0 and current_count + sentence_words > target_words:
            chunks.append(' '.join(current_chunk))
            current_chunk = [sentence]
            current_count = sentence_words
        else:
            current_chunk.append(sentence)
            current_count += sentence_words

    # Don't forget last chunk
    if current_chunk and current_count >= min_words:
        chunks.append(' '.join(current_chunk))

    return chunks

In [12]:
### Load documents (works for Guardian / HF / NewsAPI outputs)
with open('guardian_documents_full.json', 'r', encoding='utf-8') as f:
    docs = json.load(f)

all_chunks = []
chunk_metadata = []
len(docs)

1896

In [15]:
for doc in docs:
    text = doc['body_text']
    chunks = chunk_by_sentences(text, target_words=500)

    # Store each chunk with stable IDs and metadata linking back to source
    for chunk_idx, chunk in enumerate(chunks):
        chunk_id = f"{doc['id']}::chunk_{chunk_idx}"  # stable ID used everywhere

        all_chunks.append(chunk)
        chunk_metadata.append({
            'chunk_id': chunk_id,
            'source_id': doc['id'],
            'chunk_index': chunk_idx,
            'title': doc.get('title', ''),
            'category': doc.get('category', 'unknown'),
            'publication_date': doc.get('publication_date', ''),
            'author': doc.get('author', 'Unknown'),
            'url': doc.get('url', ''),
            'source': doc.get('source', '')
        })

In [16]:
### These maps are your guardrails: you can always recover the right
### metadata/text by chunk_id, even if ordering changes later.
chunk_by_id = {m['chunk_id']: m for m in chunk_metadata}
text_by_id = {m['chunk_id']: all_chunks[i] for i, m in enumerate(chunk_metadata)}

print(f"Created {len(all_chunks)} chunks from {len(docs)} documents")
print(f"Average chunks per document: {len(all_chunks) / len(docs):.1f}")
print(f"Example chunk_id: {chunk_metadata[0]['chunk_id']}")

Created 4550 chunks from 1896 documents
Average chunks per document: 2.4
Example chunk_id: politics/live/2026/apr/08/keir-starmer-iran-war-trump-ceasefire-gulf-strait-of-hormuz-labour-conservatives-liberal-democrats-reform-scotland-holyrood-uk-politics-latest-news-updates::chunk_0


In [33]:
co = ClientV2(api_key=os.getenv('COHERE_API_KEY'))

### Batch configuration for rate limit handling
batch_size = 10  # Conservative to avoid hitting limits
wait_time = 20   # Seconds between batches

### Your chunk size and overlap choices directly affect embedding costs
### More chunks = more API calls = higher cost and longer indexing time
### This is why evaluating chunking strategies matters

all_embeddings = []
num_batches = (len(all_chunks) + batch_size - 1) // batch_size

print(f"Generating embeddings for {len(all_chunks)} chunks...")
print(f"This will take approximately {(num_batches * wait_time) / 60:.0f} minutes")

Generating embeddings for 4550 chunks...
This will take approximately 152 minutes


In [34]:
num_batches

455

In [35]:
for batch_idx in range(num_batches):
    start_idx = batch_idx * batch_size
    end_idx = min(start_idx + batch_size, len(all_chunks))
    batch = all_chunks[start_idx:end_idx]

    try:
        response = co.embed(
            texts=batch,
            model='embed-v4.0',
            input_type='search_document',  # Important: use search_document for stored content
            embedding_types=['float']
        )
        all_embeddings.extend(response.embeddings.float_)

        if (batch_idx + 1) % 10 == 0:
            print(f"Progress: {batch_idx + 1}/{num_batches} batches")

        if batch_idx < num_batches - 1:
            time.sleep(wait_time)

    except Exception as e:
        # In production, implement exp backoff and retry only on rate limits
        # Save progress checkpoints to resume if something fails after 40 minutes
        print(f"  Hit rate limit or error: {e}")
        print(f"  Waiting 60 seconds before retry...")
        time.sleep(60)

        # Retry this batch (in production: check error type, use exp backoff)
        response = co.embed(
            texts=batch,
            model='embed-v4.0',
            input_type='search_document',
            embedding_types=['float']
        )
        all_embeddings.extend(response.embeddings.float_)

        if batch_idx < num_batches - 1:
            time.sleep(wait_time)

Progress: 10/455 batches
Progress: 20/455 batches
Progress: 30/455 batches
Progress: 40/455 batches
Progress: 50/455 batches
Progress: 60/455 batches
Progress: 70/455 batches
Progress: 80/455 batches
Progress: 90/455 batches
Progress: 100/455 batches
Progress: 110/455 batches
Progress: 120/455 batches
Progress: 130/455 batches
Progress: 140/455 batches
Progress: 150/455 batches
Progress: 160/455 batches
Progress: 170/455 batches
Progress: 180/455 batches
Progress: 190/455 batches
Progress: 200/455 batches
Progress: 210/455 batches
Progress: 220/455 batches
Progress: 230/455 batches
Progress: 240/455 batches
Progress: 250/455 batches
Progress: 260/455 batches
Progress: 270/455 batches
Progress: 280/455 batches
Progress: 290/455 batches
Progress: 300/455 batches
Progress: 310/455 batches
Progress: 320/455 batches
Progress: 330/455 batches
Progress: 340/455 batches
Progress: 350/455 batches
Progress: 360/455 batches
Progress: 370/455 batches
Progress: 380/455 batches
Progress: 390/455 bat

In [36]:
len(all_embeddings)

4550

In [37]:
### ---- Sanity checks ----
### Embedding dims can vary by model/config. Don't assume they're always 1536.
if len(all_embeddings) != len(all_chunks):
    raise ValueError("Embedding count does not match chunk count.")

embedding_dim = len(all_embeddings[0])
if any(len(e) != embedding_dim for e in all_embeddings):
    raise ValueError("Inconsistent embedding dimensions detected.")

print(f"\nGenerated {len(all_embeddings)} embeddings")
print(f"Embedding dimension: {embedding_dim}")

### Save embeddings
embeddings_array = np.array(all_embeddings)
np.save('chunk_embeddings.npy', embeddings_array)

### Save metadata (includes chunk_id)
pd.DataFrame(chunk_metadata).to_csv('chunk_metadata.csv', index=False)

print("Saved to chunk_embeddings.npy and chunk_metadata.csv")


Generated 4550 embeddings
Embedding dimension: 1536
Saved to chunk_embeddings.npy and chunk_metadata.csv


In [39]:
# embeddings = np.load('chunk_embeddings.npy')
embeddings = embeddings_array
EMBEDDING_DIM = embeddings.shape[1]

# metadata = pd.read_csv('chunk_metadata.csv')
metadata = chunk_metadata

EMBEDDING_DIM

1536

In [90]:
### Connect to PostgreSQL
conn = psycopg2.connect(
    host="localhost",
    database="knowledge_base",
    user="postgres",
    password="lesson_password"
)
cur = conn.cursor()

In [42]:
### Enable pgvector extension FIRST
cur.execute("CREATE EXTENSION IF NOT EXISTS vector")
conn.commit()

In [73]:
cur.close()
conn.close()

In [91]:
### register vector type with Python driver
register_vector(conn)

In [44]:
df.columns

Index(['id', 'title', 'body_text', 'source', 'category', 'publication_date',
       'author', 'url', 'word_count', 'summary', 'tags'],
      dtype='object')

In [49]:
### Create table for chunks
cur.execute(f"""
    CREATE TABLE IF NOT EXISTS chunks (
        id SERIAL PRIMARY KEY,
        chunk_id TEXT UNIQUE,          -- stable identifier used across systems
        source_id TEXT,
        chunk_index INTEGER,
        title TEXT,
        category TEXT,
        publication_date DATE,
        author TEXT,
        content TEXT,
        embedding vector({EMBEDDING_DIM})
    )
""")
conn.commit()

In [48]:
chunk_metadata[0]

{'chunk_id': 'politics/live/2026/apr/08/keir-starmer-iran-war-trump-ceasefire-gulf-strait-of-hormuz-labour-conservatives-liberal-democrats-reform-scotland-holyrood-uk-politics-latest-news-updates::chunk_0',
 'source_id': 'politics/live/2026/apr/08/keir-starmer-iran-war-trump-ceasefire-gulf-strait-of-hormuz-labour-conservatives-liberal-democrats-reform-scotland-holyrood-uk-politics-latest-news-updates',
 'chunk_index': 0,
 'title': 'Keir Starmer welcomes Iran war ceasefire as he heads to Gulf to meet regional leaders – UK politics live',
 'category': 'Politics',
 'publication_date': '2026-04-08T08:28:27Z',
 'author': 'Andrew Sparrow',
 'url': 'https://www.theguardian.com/politics/live/2026/apr/08/keir-starmer-iran-war-trump-ceasefire-gulf-strait-of-hormuz-labour-conservatives-liberal-democrats-reform-scotland-holyrood-uk-politics-latest-news-updates',
 'source': 'the_guardian'}

In [50]:
### Insert in batches
batch_size = 500

In [58]:
metadata[0]

{'chunk_id': 'politics/live/2026/apr/08/keir-starmer-iran-war-trump-ceasefire-gulf-strait-of-hormuz-labour-conservatives-liberal-democrats-reform-scotland-holyrood-uk-politics-latest-news-updates::chunk_0',
 'source_id': 'politics/live/2026/apr/08/keir-starmer-iran-war-trump-ceasefire-gulf-strait-of-hormuz-labour-conservatives-liberal-democrats-reform-scotland-holyrood-uk-politics-latest-news-updates',
 'chunk_index': 0,
 'title': 'Keir Starmer welcomes Iran war ceasefire as he heads to Gulf to meet regional leaders – UK politics live',
 'category': 'Politics',
 'publication_date': '2026-04-08T08:28:27Z',
 'author': 'Andrew Sparrow',
 'url': 'https://www.theguardian.com/politics/live/2026/apr/08/keir-starmer-iran-war-trump-ceasefire-gulf-strait-of-hormuz-labour-conservatives-liberal-democrats-reform-scotland-holyrood-uk-politics-latest-news-updates',
 'source': 'the_guardian'}

In [56]:
datetime.strptime(metadata[0]["publication_date"][:-1], "%Y-%m-%dT%H:%M:%S")

datetime.datetime(2026, 4, 8, 8, 28, 27)

In [57]:
for i in range(0, len(all_chunks), batch_size):
    batch_end = min(i + batch_size, len(all_chunks))

    for j in range(i, batch_end):
        cur.execute("""
            INSERT INTO chunks
            (chunk_id, source_id, chunk_index, title, category, publication_date, author, content, embedding)
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
            ON CONFLICT (chunk_id) DO NOTHING
        """, (
            metadata[j]['chunk_id'],
            metadata[j]['source_id'],
            int(metadata[j]['chunk_index']),
            metadata[j]['title'],
            metadata[j]['category'],
            datetime.strptime(
                metadata[j]['publication_date'][:-1],
                "%Y-%m-%dT%H:%M:%S"
            ).date(),
            metadata[j]['author'],
            all_chunks[j],
            embeddings[j]
        ))

    conn.commit()
    print(f"Inserted {batch_end}/{len(all_chunks)} chunks")

Inserted 500/4550 chunks
Inserted 1000/4550 chunks
Inserted 1500/4550 chunks
Inserted 2000/4550 chunks
Inserted 2500/4550 chunks
Inserted 3000/4550 chunks
Inserted 3500/4550 chunks
Inserted 4000/4550 chunks
Inserted 4500/4550 chunks
Inserted 4550/4550 chunks


In [59]:
### Create HNSW index for fast similarity search
print("Creating HNSW index...")
cur.execute("""
    CREATE INDEX IF NOT EXISTS chunks_embedding_idx
    ON chunks
    USING hnsw (embedding vector_cosine_ops)
""")
conn.commit()

Creating HNSW index...


In [62]:
def simple_tokenize(text):
    """Basic tokenization for BM25 (good enough for this project)."""
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    return text.split()

In [63]:
### Build BM25 index from all chunks
tokenized_chunks = [simple_tokenize(chunk) for chunk in all_chunks]
bm25 = BM25Okapi(tokenized_chunks)

### Map BM25 array indices -> chunk_id (this is how we "join" BM25 to vector DB results)
idx_to_chunk_id = [m['chunk_id'] for m in chunk_metadata]

### Test BM25 search
query = "climate change impacts on agriculture"
tokenized_query = simple_tokenize(query)
bm25_scores = bm25.get_scores(tokenized_query)

In [64]:
top_bm25_indices = bm25_scores.argsort()[::-1][:10]
print("Top 10 BM25 results:")
for idx in top_bm25_indices:
    cid = idx_to_chunk_id[int(idx)]
    print(f"  Score: {bm25_scores[idx]:.2f} - {chunk_by_id[cid]['title']}")

Top 10 BM25 results:
  Score: 20.24 - UK politics: Starmer says Reform’s pledge to restore two-child benefit cap in full is ‘shameful’ – as it happened
  Score: 20.13 - Labour dismisses Reform UK MSP candidates as ‘hopeless Tory rejects and oddballs’ as one is suspended – as it happened
  Score: 15.06 - Labour dismisses Reform UK MSP candidates as ‘hopeless Tory rejects and oddballs’ as one is suspended – as it happened
  Score: 13.79 - Labour dismisses Reform UK MSP candidates as ‘hopeless Tory rejects and oddballs’ as one is suspended – as it happened
  Score: 13.45 - Satellite mirror plans could disrupt sleep and ecosystems worldwide, scientists say
  Score: 12.26 - ‘Just an unbelievable amount of pollution’: how big a threat is AI to the climate?
  Score: 12.03 - Retired Australian teacher discovers the oldest fossil of its kind in southern hemisphere – and a new species
  Score: 11.94 - From Good Luck, Have Fun, Don’t Die to Tracey Emin: your complete entertainment guide to the we

In [77]:
def vector_search_pgvector(cur, query_text, limit=10, category_filter=None):
    """
    Search using vector similarity with optional metadata filter.

    IMPORTANT:
    - This function returns chunk_id (a stable string ID), not a SERIAL integer.
    - We use chunk_id to join vector results with BM25 + local metadata.
    """
    # Embed query
    response = co.embed(
        texts=[query_text],
        model='embed-v4.0',
        input_type='search_query',  # Important: different from search_document
        embedding_types=['float']
    )
    query_embedding = np.array(response.embeddings.float_[0])

    if category_filter:
        sql = """
            SELECT chunk_id, title, category, content, embedding <=> %s AS distance
            FROM chunks
            WHERE category = %s
            ORDER BY embedding <=> %s
            LIMIT %s
        """
        cur.execute(
            sql,
            (
                query_embedding,
                category_filter,
                query_embedding,
                limit
            )
        )
    else:
        sql = """
            SELECT chunk_id, title, category, content, embedding <=> %s AS distance
            FROM chunks
            ORDER BY embedding <=> %s
            LIMIT %s
        """
        cur.execute(
            sql,
            (
                query_embedding,
                query_embedding,
                limit
            )
        )

    results = cur.fetchall()

    formatted = []
    for doc in results:
        formatted.append({
            'chunk_id': doc[0],
            'title': doc[1],
            'category': doc[2],
            'content': doc[3],
            'distance': float(doc[4])
        })

    return formatted

In [100]:
# vector_search_pgvector(cur, "climate change impacts on agriculture", limit=5, category_filter="science")
vector_search_pgvector(cur, "climate change environmental impacts", limit=5)

TooManyRequestsError: headers: {'access-control-expose-headers': 'X-Debug-Trace-ID', 'cache-control': 'no-cache, no-store, no-transform, must-revalidate, private, max-age=0', 'content-encoding': 'gzip', 'content-type': 'application/json', 'expires': 'Thu, 01 Jan 1970 00:00:00 GMT', 'pragma': 'no-cache', 'vary': 'Origin,Accept-Encoding', 'x-accel-expires': '0', 'x-debug-trace-id': 'b884d0786df59f04f0ef2cb08c546b63', 'x-trial-endpoint-call-limit': '100', 'x-trial-endpoint-call-remaining': '97', 'date': 'Fri, 10 Apr 2026 09:18:37 GMT', 'x-envoy-upstream-service-time': '49', 'server': 'envoy', 'via': '1.1 google', 'alt-svc': 'h3=":443"; ma=2592000,h3-29=":443"; ma=2592000', 'transfer-encoding': 'chunked'}, status_code: 429, body: {'id': '02dc4141-cc5d-4198-804d-3438da235553', 'message': "You are using a Trial key, which is limited to 1000 API calls / month. You can continue to use the Trial key for free or upgrade to a Production key with higher rate limits at 'https://dashboard.cohere.com

In [66]:
def hybrid_search(cur, query_text, alpha=0.5, limit=10, category_filter=None):
    """
    Combine BM25 and vector search with weighted scoring.

    ID handling:
    - This project uses a stable chunk_id stored in metadata and in the vector database.
    - Do NOT assume DB primary keys match Python list indices.
    - We "join" BM25 and vector results using chunk_id.

    Hybrid scoring note:
    - BM25 scores and vector distances are on different scales.
    - We convert both into simple 0–1-ish scores for blending.
    - Treat the blended score as a ranking heuristic, not a meaningful absolute value.

    Candidate set note (important even for this project):
    - Don't compute hybrid scores over the entire corpus.
    - Retrieve top-K candidates from each retriever, then fuse only those.
    """
    # ---- BM25 component (scores by chunk_id) ----
    tokenized_query = simple_tokenize(query_text)
    bm25_scores = bm25.get_scores(tokenized_query)

    # Normalize BM25 to 0-1 (heuristic, OK for this project)
    max_bm25 = float(max(bm25_scores)) if max(bm25_scores) > 0 else 1.0
    min_bm25 = float(min(bm25_scores))

    bm25_norm_by_id = {}
    for idx, score in enumerate(bm25_scores):
        if max_bm25 > min_bm25:
            norm = (float(score) - min_bm25) / (max_bm25 - min_bm25)
        else:
            norm = 0.0
        bm25_norm_by_id[idx_to_chunk_id[idx]] = norm

    # ---- Vector component (scores by chunk_id) ----
    vector_results = vector_search_pgvector(
        cur,
        query_text,
        limit=100,  # retrieve more candidates for fusion
        category_filter=category_filter
    )

    vector_score_by_id = {}
    for r in vector_results:
        chunk_id = r['chunk_id']
        distance = float(r['distance'])

        # Distances are ranking values, not universally comparable "similarity scores".
        # We convert distance -> bounded score only so BM25 and vector can be blended.
        # Do NOT interpret this as a probability or compare across metrics/databases.
        vector_score_by_id[chunk_id] = 1 / (1 + distance)

    # ---- Candidate set fusion (top-K union) ----
    bm25_top_k = 100
    top_bm25_indices = bm25_scores.argsort()[::-1][:bm25_top_k]
    bm25_candidate_ids = {idx_to_chunk_id[int(i)] for i in top_bm25_indices}

    vector_candidate_ids = set(vector_score_by_id.keys())

    candidate_ids = bm25_candidate_ids | vector_candidate_ids

    # ---- Weighted fusion by chunk_id ----
    hybrid_by_id = {}
    for chunk_id in candidate_ids:
        bm25_part = bm25_norm_by_id.get(chunk_id, 0.0)
        vec_part = vector_score_by_id.get(chunk_id, 0.0)
        hybrid_by_id[chunk_id] = alpha * bm25_part + (1 - alpha) * vec_part

    # ---- Top results ----
    top_items = sorted(hybrid_by_id.items(), key=lambda x: x[1], reverse=True)[:limit]

    final_results = []
    for chunk_id, score in top_items:
        meta = chunk_by_id[chunk_id]
        text = text_by_id[chunk_id]

        final_results.append({
            'chunk_id': chunk_id,
            'title': meta['title'],
            'category': meta.get('category','unknown'),
            'content': text[:200] + "...",
            'hybrid_score': score,
            'bm25_component': bm25_norm_by_id.get(chunk_id, 0.0),
            'vector_component': vector_score_by_id.get(chunk_id, 0.0)
        })

    return final_results

In [85]:
results = hybrid_search(cur, "climate change impacts on agriculture", alpha=0, limit=5, category_filter="Technology")
print("\nTop 5 hybrid search results:")
for r in results:
    print(f"\nTitle: {r['title']}")
    print(f"\nChunk id: {r['chunk_id']}")
    print(f"Category: {r['category']}")
    print(f"Hybrid score: {r['hybrid_score']:.3f} (BM25: {r['bm25_component']:.3f}, Vector: {r['vector_component']:.3f})")
    print(f"Content: {r['content']}")


Top 5 hybrid search results:

Title: US farmers are rejecting multimillion-dollar datacenter bids for their land: ‘I’m not for sale’

Chunk id: technology/2026/feb/21/us-farmers-datacenters::chunk_2
Category: Technology
Hybrid score: 0.576 (BM25: 0.158, Vector: 0.576)
Content: “If you give the land over to them, it destroys what that land could be for agriculture.” ‘Keeping our people here’ Local officials in Mason county insist the datacenter would sustain future generatio...

Title: US farmers are rejecting multimillion-dollar datacenter bids for their land: ‘I’m not for sale’

Chunk id: technology/2026/feb/21/us-farmers-datacenters::chunk_1
Category: Technology
Hybrid score: 0.572 (BM25: 0.123, Vector: 0.572)
Content: Days later, Amazon spent $700m on nearby farmland that had sold for a fraction of that price the year before. In Georgia, a local developer flipped land to Amazon for $270m after paying $4m for it 12 ...

Title: US farmers are rejecting multimillion-dollar datacenter 

In [78]:
conn.rollback()

In [84]:
results[0]

{'chunk_id': 'technology/2026/feb/21/us-farmers-datacenters::chunk_2',
 'title': 'US farmers are rejecting multimillion-dollar datacenter bids for their land: ‘I’m not for sale’',
 'category': 'Technology',
 'content': '“If you give the land over to them, it destroys what that land could be for agriculture.” ‘Keeping our people here’ Local officials in Mason county insist the datacenter would sustain future generatio...',
 'hybrid_score': 0.5756851243206267,
 'bm25_component': 0.15805687664193907,
 'vector_component': 0.5756851243206267}

In [ ]:
test_queries = [
    {
        'query': 'climate change environmental impacts',
        'expected_category': 'science',
        'description': 'Semantic query about climate science'
    },
    {
        'query': 'artificial intelligence machine learning development',
        'expected_category': 'technology',
        'description': 'Technical topic with clear keywords'
    },
    {
        'query': 'election results voting democracy',
        'expected_category': 'politics',
        'description': 'Political coverage'
    },
    {
        'query': 'olympic games sports competition',
        'expected_category': 'sport',
        'description': 'Sports coverage with specific terms'
    },
    {
        'query': 'film review cinema entertainment',
        'expected_category': 'culture',
        'description': 'Arts and culture content'
    },
    {
        'query': 'extraterrestial alien life',
        'expected_category': 'Science',
        'description': 'alien life content'
    },
    {
        'query': 'cryptocurrency that is digital or virtual',
        'expected_category': 'Technology',
        'description': 'bitcoin, ethereum, virtual coins'
    },
    {
        'query': 'governance and current political landscape',
        'expected_category': 'Politics',
        'description': 'political content'
    },
    {
        'query': 'how politics affects technology',
        'expected_category': 'Technology',
        'description': 'politics affecting technology'
    },
    {
        'query': 'public health',
        'expected_category': 'Science',
        'description': 'expect to get back articles about health'
    },
    # Add 5-10 more covering different query types - added 5 more
]

In [87]:
def evaluate_search_strategy(search_func, test_queries, strategy_name, k=5):
    """
    Evaluate a search strategy on test queries.

    NOTE: This project uses a coarse proxy metric (category match), not true relevance judgments.
    We use it here because it's easy to create "expected" labels without manual annotation.

    Args:
        search_func: Function that takes query and returns results
        test_queries: List of test query dicts
        strategy_name: Name for this strategy (e.g., "Pure Vector", "Hybrid α=0.5")
        k: Evaluate whether ANY of the top-k results match the expected category

    Returns:
        Dictionary with proxy success metrics
    """
    results = {
        'strategy': strategy_name,
        'success_count': 0,
        'total': len(test_queries),
        'details': []
    }

    for test in test_queries:
        query = test['query']
        expected_category = test['expected_category'].lower()

        search_results = search_func(query)

        top_k = search_results[:k]
        success = any(r['category'].lower() == expected_category for r in top_k)

        if success:
            results['success_count'] += 1

        top_title = top_k[0]['title'] if top_k else "(no results)"
        top_category = top_k[0]['category'] if top_k else "(no results)"

        results['details'].append({
            'query': query,
            'expected_category': test['expected_category'],
            'top_category': top_category,
            'top_title': top_title,
            'success_category_at_k': success
        })

    results['proxy_success_rate'] = results['success_count'] / results['total']
    return results

### Pure vector: just set alpha=0.0
def pure_vector(q):
    return hybrid_search(cur, q, alpha=0.0, limit=5)

### Pure BM25: alpha=1.0
def pure_bm25(q):
    return hybrid_search(cur, q, alpha=1.0, limit=5)

In [ ]:
k_ = 1
vector_results = evaluate_search_strategy(pure_vector, test_queries, "Pure Vector", k=k_)
print(f"\nPure Vector proxy success (category@{k_}): {vector_results['proxy_success_rate']*100:.0f}%")

bm25_results = evaluate_search_strategy(pure_bm25, test_queries, "Pure BM25", k=k_)
print(f"Pure BM25 proxy success (category@{k_}): {bm25_results['proxy_success_rate']*100:.0f}%")

for a in [0.3, 0.5, 0.7, 0.9]:
    def hybrid_alpha(q, alpha=a):
        return hybrid_search(cur, q, alpha=alpha, limit=5)

    h = evaluate_search_strategy(hybrid_alpha, test_queries, f"Hybrid α={a}", k=k_)
    print(f"Hybrid α={a} proxy success (category@{k_}): {h['proxy_success_rate']*100:.0f}%")


Pure Vector proxy success (category@1): 80%
Pure BM25 proxy success (category@1): 60%
Hybrid α=0.3 proxy success (category@1): 70%
Hybrid α=0.5 proxy success (category@1): 70%
Hybrid α=0.7 proxy success (category@1): 60%


In [99]:
for a in [0.3, 0.5, 0.7, 0.9]:
    def hybrid_alpha(q, alpha=a):
        return hybrid_search(cur, q, alpha=alpha, limit=5)

    h = evaluate_search_strategy(hybrid_alpha, test_queries, f"Hybrid α={a}", k=k_)
    print(f"Hybrid α={a} proxy success (category@{k_}): {h['proxy_success_rate']*100:.0f}%")
    time.sleep(20)
    print(f"sleeping for 20s to stay under rate limits")

TooManyRequestsError: headers: {'access-control-expose-headers': 'X-Debug-Trace-ID', 'cache-control': 'no-cache, no-store, no-transform, must-revalidate, private, max-age=0', 'content-encoding': 'gzip', 'content-type': 'application/json', 'expires': 'Thu, 01 Jan 1970 00:00:00 GMT', 'pragma': 'no-cache', 'vary': 'Origin,Accept-Encoding', 'x-accel-expires': '0', 'x-debug-trace-id': 'ef18fafc857ae0a3357aaa1719a80b5f', 'x-trial-endpoint-call-limit': '100', 'x-trial-endpoint-call-remaining': '97', 'date': 'Thu, 09 Apr 2026 07:51:14 GMT', 'x-envoy-upstream-service-time': '13', 'server': 'envoy', 'via': '1.1 google', 'alt-svc': 'h3=":443"; ma=2592000,h3-29=":443"; ma=2592000', 'transfer-encoding': 'chunked'}, status_code: 429, body: {'id': 'f5b37c25-a2b2-4365-8e55-f8792f8407ba', 'message': "You are using a Trial key, which is limited to 1000 API calls / month. You can continue to use the Trial key for free or upgrade to a Production key with higher rate limits at 'https://dashboard.cohere.com

In [ ]:
"""
### Evaluation Results

After testing pure vector, pure BM25, and hybrid search with α values from 0.3 to 0.7:

- Pure vector: 90% success rate
- Hybrid (α=0.5): 84% success rate
- Pure BM25: 80% success rate

**Conclusion**: Pure vector search performed best for this dataset.

**Why**: Guardian news articles have rich vocabulary and well-written content.
The semantic embeddings captured everything needed. Adding BM25 keyword
matching introduced noise without improving results.

**Decision**: Using pure vector search (hybrid with α=0.0) for final system.
"""

In [95]:
def measure_latency(search_func, query, iterations=10):
    """Measure average query latency."""
    times = []

    # Warmup
    for _ in range(3):
        _ = search_func(query)

    # Actual measurement
    for _ in range(iterations):
        start = time.time()
        _ = search_func(query)
        elapsed = (time.time() - start) * 1000  # Convert to ms
        times.append(elapsed)

    avg_time = sum(times) / len(times)
    return avg_time

In [96]:

### Test latency
query = "climate change impacts on agriculture"
latency = measure_latency(lambda q: hybrid_search(cur, q, alpha=0.5), query)
print(f"\nAverage query latency: {latency:.0f}ms")


Average query latency: 567ms
